# MVPA — 01: Compute Temporal Decoding

Runs temporal decoding (and optionally temporal generalization and cross-decoding)
for one or more condition pairs.

**Warning:** this is computationally expensive. Use `n_jobs=-1` and start with
fewer repeats (`n_repeats=10`) to test before running the full analysis.

**Output:** per-subject decoding scores saved to the analysis directory

In [ ]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import load_config, decode_all, load_all_scores

# ── Update this path ──
cfg = load_config('../../configs/your_experiment.yaml')

print("Setup OK")

In [ ]:
# ── Define conditions to decode ──
# Keys = condition labels (used for output filenames and plots)
# Values = list of event codes belonging to that condition
# ── Update event codes to match your experiment ──
conditions = {
    'condition_a': [10, 11, 12],   # replace with your event codes
    'condition_b': [20, 21, 22],
}

summary, group = decode_all(
    cfg,
    window_name='your_window',        # must match an epoching window name
    conditions=conditions,
    analysis_name='a_vs_b',           # used in output filenames
    classifier='linear_svc',
    n_folds=3,
    n_repeats=100,                    # reduce for testing
    resample_sfreq=256,               # set to None to skip resampling
    baseline=(-0.2, 0.0),             # set to None to skip baseline
    n_null_permutations=1,            # increase for publication (e.g. 100)
    compute_distances=True,
    n_jobs=-1,
    overwrite=False,
)

## Alternative: Label-based decoding

When labels come from behavioral data (e.g., accuracy, congruency, stimulus identity) rather than event codes, use `decode_subject` with the `labels` parameter. This requires a manual loop over subjects since each subject needs its own behavioral alignment.

Use `align_epochs_behavior()` to match your behavioral DataFrame with the surviving epochs via the drop log.

In [ ]:
# ── Optional: label-based decoding (from behavioral data) ──
# Uncomment and adapt this cell if your labels come from a behavioral
# DataFrame rather than from event codes.

# import pandas as pd
# from eeg_toolkit.mvpa import decode_subject, align_epochs_behavior
# from eeg_toolkit import load_config, get_subjects
#
# cfg = load_config('../../configs/your_experiment.yaml')
# beh = pd.read_csv('path/to/behavioral_data.csv')
#
# for subject in get_subjects(cfg):
#     # 1. Load epochs to align with behavioral data
#     import mne
#     from eeg_toolkit import get_epochs_path
#     epochs = mne.read_epochs(
#         get_epochs_path(cfg, subject, 'your_window'), preload=True
#     )
#
#     # 2. Align behavioral data with surviving epochs
#     subj_beh = beh[beh['subject'] == subject].reset_index(drop=True)
#     beh_aligned = align_epochs_behavior(epochs, subj_beh)
#     del epochs  # decode_subject will reload them
#
#     # 3. Extract labels from behavioral column
#     labels = beh_aligned['your_column'].values
#
#     # 4. Optional: filter trials with a boolean mask
#     # trial_mask = beh_aligned['accuracy'] == 1  # e.g., correct only
#     # labels = labels[trial_mask]
#
#     # 5. Decode
#     ok, result = decode_subject(
#         cfg, subject,
#         window_name='your_window',
#         analysis_name='my_label_analysis',
#         labels=labels,
#         # trial_mask=trial_mask,     # uncomment if filtering
#         # chance_level=0.25,         # set if multiclass (auto-computed if None)
#         classifier='linear_svc',
#         n_folds=3,
#         n_repeats=100,
#         resample_sfreq=256,
#         baseline=(-0.2, 0.0),
#         n_null_permutations=1,
#         overwrite=False,
#     )
#     print(f"{subject}: {'OK' if ok else 'skipped'}")

## Alternative: Multiclass decoding

The toolkit supports any number of classes (not just binary). Simply pass more than 2 conditions. Chance level is auto-computed as `1/n_classes`.

In [ ]:
# ── Optional: multiclass decoding (e.g., 4 serial positions) ──
# Uncomment and adapt this cell for multiclass classification.

# conditions_multi = {
#     'pos1': [5, 9],     # event codes for position 1
#     'pos2': [6, 10],    # event codes for position 2
#     'pos3': [7, 11],    # event codes for position 3
#     'pos4': [8, 12],    # event codes for position 4
# }
#
# summary_mc, group_mc = decode_all(
#     cfg,
#     window_name='your_window',
#     conditions=conditions_multi,
#     analysis_name='position_4class',
#     # chance_level is auto-computed as 0.25 for 4 classes
#     classifier='linear_svc',
#     n_folds=3,
#     n_repeats=10,
#     resample_sfreq=256,
#     baseline=(-0.2, 0.0),
#     n_null_permutations=1,
#     compute_distances=False,   # not supported for multiclass
#     n_jobs=-1,
#     overwrite=False,
# )

In [ ]:
# ── Optional: temporal generalization matrix ──
# Trains a classifier at each time point and tests it at all other time points.
# Slower — reduce resample_sfreq (e.g. 128) and n_repeats (e.g. 1).
summary_tg, group_tg = decode_all(
    cfg,
    window_name='your_window',
    conditions=conditions,
    analysis_name='a_vs_b_tg',
    classifier='linear_svc',
    n_folds=3,
    n_repeats=1,
    resample_sfreq=128,
    baseline=(-0.2, 0.0),
    compute_temporal_gen=True,
    n_jobs=-1,
    overwrite=False,
)

In [ ]:
# ── Optional: cross-decoding ──
# Train on one set of conditions, test on another.
from eeg_toolkit.mvpa import cross_decode_all

train_conditions = {
    'left':  [10, 11],
    'right': [12, 13],
}
test_conditions = {
    'left':  [20, 21],
    'right': [22, 23],
}

summary_xdec, group_xdec = cross_decode_all(
    cfg,
    window_name='your_window',
    train_conditions=train_conditions,
    test_conditions=test_conditions,
    analysis_name='cross_decode',
    classifier='linear_svc',
    resample_sfreq=256,
    baseline=(-0.2, 0.0),
    bidirectional=True,
    n_null_permutations=1,
    n_jobs=-1,
    overwrite=False,
)